# 04 — Self-Play: Critic-Guided Dialogue Refinement

Iteratively improves the SFT model's therapy skills through self-play:

1. **Generate** 112 simulated doctor–patient dialogues using CBT-Bench vignettes
2. **Critic** (DeepSeek) evaluates therapist performance using CTRS-R rubric + moderator decides when to stop
3. **Refine** — therapist regenerates the dialogue from scratch using critic feedback
4. Repeat critic→refine cycle **2 times** (configurable)
5. **Collect** the final refined dialogues as SFT training data

The outer loop repeats this whole process `N_OUTER_LOOPS` times to accumulate data.

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")

import asyncio
import json
import os
import re
import time
from pathlib import Path

import torch
from datasets import load_dataset
from openai import AsyncOpenAI
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f\"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\")"

## Configuration

In [ ]:
# ── Therapist model ───────────────────────────────────────────────────────────
BASE_MODEL_NAME = "Qwen/Qwen3.5-4B"

SSD_ROOT        = Path(os.environ.get("SSD_ROOT", "/tmp"))
SFT_MODEL_PATH  = SSD_ROOT / "models/Qwen3.5-4B-SFT/lora-adapters"

# ── DeepSeek models ───────────────────────────────────────────────────────────
PATIENT_MODEL    = "deepseek-v4-flash"
CRITIC_MODEL     = "deepseek-v4-flash"
MODERATOR_MODEL  = "deepseek-v4-flash"
MIN_TURNS        = 4     # minimum exchanges before moderator can end session

# ── Conversation ─────────────────────────────────────────────────────────────
MAX_TURNS             = 20    # hard cap on therapist-patient exchanges
THERAPIST_MAX_TOKENS  = 1000
PATIENT_MAX_TOKENS    = 1000
CRITIC_MAX_TOKENS     = 50000

# ── Self-play ─────────────────────────────────────────────────────────────────
N_REFINEMENT_CYCLES = 2       # number of critic→refine iterations per outer loop
N_OUTER_LOOPS       = 1       # how many times to repeat the full process

# ── Concurrency ──────────────────────────────────────────────────────────────
API_CONCURRENCY = 500   # semaphore cap for DeepSeek calls

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SELF_PLAY_JSONL = OUTPUT_DIR / "self_play_finetune.jsonl"

# ── Therapist system prompt ───────────────────────────────────────────────────
THERAPIST_SYSTEM = (
    "You are a skilled CBT (Cognitive Behavioural Therapy) therapist conducting an initial "
    "clinical assessment session. Follow CBT principles:\n"
    "- Open with a collaborative agenda-setting check-in\n"
    "- Use Socratic questioning to explore thoughts, feelings, and behaviours\n"
    "- Help the patient identify links between cognitions, emotions, and behaviours\n"
    "- Validate emotions while gently challenging unhelpful thought patterns\n"
    "- Work toward a shared conceptualisation and practical goal-oriented interventions\n"
    "- Be warm, empathic, genuine, and professionally boundaried\n"
    "- Propose appropriate between-session homework toward the end of the session\n\n"
    "Respond as a therapist — ask thoughtful questions, reflect back, guide toward insight. "
    "Keep responses focused (2–4 sentences)."
)

print(f"SFT model       : {SFT_MODEL_PATH}")
print(f"Refinement cycles: {N_REFINEMENT_CYCLES}")
print(f"Outer loops      : {N_OUTER_LOOPS}")
print(f"Concurrency      : {API_CONCURRENCY}")
print(f"Output           : {SELF_PLAY_JSONL}")

## Clinical Vignettes — CBT-Bench `core_fine_test`

In [ ]:
cbt_bench = load_dataset("Psychotherapy-LLM/CBT-Bench", "core_fine_test", split="train")
print(f"Loaded {len(cbt_bench)} examples from CBT-Bench core_fine_test")


def row_to_vignette(row: dict, idx: int) -> dict:
    """Convert a CBT-Bench core_fine_test row into a patient vignette."""
    beliefs = ", ".join(row["core_belief_fine_grained"])

    background = (
        f"Situation: {row['situation']}\n"
        f"Automatic thoughts: {row['thoughts']}\n"
        f"Core beliefs: {beliefs}"
    )

    system_prompt = (
        f"You are a therapy client attending a CBT session. "
        f"Here is your situation:\n{row['situation']}\n\n"
        f"Your automatic thoughts are:\n{row['thoughts']}\n\n"
        f"Your underlying core beliefs include: {beliefs}.\n\n"
        "Stay in character as this person throughout the conversation. "
        "Respond naturally — use everyday language, not clinical terms. "
        "Be open to the therapist's questions but don't volunteer your core beliefs directly; "
        "let the therapist guide you toward insight. "
        "Keep responses 2–5 sentences. Never mention therapy techniques."
    )

    return {
        "name": f"case_{row['id']}",
        "background": background,
        "system_prompt": system_prompt,
    }


VIGNETTES = [row_to_vignette(row, i) for i, row in enumerate(cbt_bench)]

print(f"\nVignettes created: {len(VIGNETTES)}")
for v in VIGNETTES[:3]:
    print(f"  {v['name']}: {v['background'][:80]}…")

## CTRS-R Rubric — 11 Items, 0–3 Scale

In [ ]:
CTSR_ITEMS = [
    {
        "number": 1, "name": "Agenda", "key": "item_1_agenda", "short_label": "Agenda",
        "criteria": (
            "Did the therapist...\n"
            "• Provide transition to the previous session?\n"
            "• Identify significant events [positive and/or negative] since previous session?\n"
            "• Review Action Plan [complete review may be done as part of the agenda]?\n"
            "• Conduct a mood check?\n"
            "• Identify specific goals or problems to work on during the session?"
        ),
        "anchors": {
            0: "If therapist completed none of the above items",
            1: "If therapist completed one or more but not all of the above items",
            2: "If therapist completed all five of the above items",
            3: "If therapist completed all five of the above items PLUS… Made certain that all items important to the client were addressed and prioritized; Followed the agenda throughout the session unless there was an overt discussion about deviating from the agenda.",
        },
    },
    {
        "number": 2, "name": "Feedback", "key": "item_2_feedback", "short_label": "Feedback",
        "criteria": (
            "Did the therapist...\n"
            "• Ascertain the client's reaction to the session, the therapist, or the therapeutic process?\n"
            "• Ensure that the client understood and agreed with the treatment plan?\n"
            "• Respond appropriately to feedback?"
        ),
        "anchors": {
            0: "If therapist completed none of the above items",
            1: "If therapist completed one or more but not all of the above items",
            2: "If therapist completed all three of the above items",
            3: "If the therapist completed all three of the above items PLUS… The therapist fluidly requested feedback throughout the session [agenda, transitions, use of techniques, and/or developing an Action Plan].",
        },
    },
    {
        "number": 3, "name": "Understanding", "key": "item_3_understanding", "short_label": "Understanding",
        "criteria": (
            "Did the therapist...\n"
            "• Demonstrate they generally heard and understood the content of what the client expressed "
            "through repeating, summarizing, etc. what the client said during the session?"
        ),
        "anchors": {
            0: "If therapist did not demonstrate the above item",
            1: "If therapist inconsistently listened and reflected the client's statements",
            2: "If therapist listened and reflected the client's statements throughout the session",
            3: "If the therapist consistently listened and reflected throughout the session PLUS… Therapist demonstrated recognition of understanding the client's emotional state through acknowledgement, reflection, empathy; Discussed the client's emotional state within the context of the conceptualization; Demonstration of emotional state is accomplished by a combination of words, expressions, gestures, tone, and body language throughout the session.",
        },
    },
    {
        "number": 4, "name": "Interpersonal Effectiveness", "key": "item_4_interpersonal_effectiveness", "short_label": "Interpersonal\nEffect.",
        "criteria": (
            "Throughout the session, did the therapist…\n"
            "• Demonstrate concern for client and help the client reach their goals?\n"
            "• Provide positive reinforcement for actions taken by the client (e.g. completing action plans)?\n"
            "• Maintain professional and ethical behavior?"
        ),
        "anchors": {
            0: "If therapist completed none of the above items",
            1: "If therapist completed one or two, but not all three of the above items",
            2: "If therapist completed all three of the above items",
            3: "If the therapist completed all three of the above items PLUS… Through words, gestures, and expressions, demonstrated warmth, genuineness, and unconditional acceptance (absence of judgment) by making positive statements about the client's character or characteristics (e.g. strength, determination, caring, vision, values, integrity, etc.)",
        },
    },
    {
        "number": 5, "name": "Collaboration", "key": "item_5_collaboration", "short_label": "Collaboration",
        "criteria": (
            "Did the therapist...\n"
            "• Ask the client for input/agreement when setting the agenda and respond appropriately to the input?\n"
            "• Ask the client for input/agreement when selecting or using CBT techniques and respond appropriately to the input?\n"
            "• Ask the client for input/agreement when determining the Action Plan to be followed between sessions and responded appropriately to the input?"
        ),
        "anchors": {
            0: "If therapist completed none of the above items",
            1: "If therapist completed one or more but not all three of the above items",
            2: "If therapist completed all three of the above items",
            3: "If the therapist completed all three of the above items PLUS… Throughout the session, the therapist made a consistent effort to invite client's participation/agreement on every major decision about the session and responded appropriately. The collaboration resulted in a mutually agreeable direction for the session.",
        },
    },
    {
        "number": 6, "name": "Pacing and Efficient Use of Time", "key": "item_6_pacing", "short_label": "Pacing",
        "criteria": (
            "Did the therapist...\n"
            "• Allocate appropriate time for transition and agenda setting; intervention(s); feedback and action planning?\n"
            "• Complete the session within 40 – 60 minutes?"
        ),
        "anchors": {
            0: "If therapist completed none of the above items",
            1: "If therapist completed one but not both of the above items",
            2: "If therapist completed both of the above items",
            3: "If the therapist completed both of the above items PLUS… Provided pacing that allowed discussion to seamlessly move through each of the different segments; AND, if needed, made appropriate attempts to limit peripheral or unproductive discussion; AND the session was conducted within 45 – 55 minutes",
        },
    },
    {
        "number": 7, "name": "Guided Discovery", "key": "item_7_guided_discovery", "short_label": "Guided\nDiscovery",
        "criteria": (
            "Did the therapist...\n"
            "• Design and conduct the session to help the client achieve a cognitive shift regarding agenda items?\n"
            "• Throughout the session, avoid showing bias and avoid use of directions, arguments, or coercion to lead the client "
            "to 'see' things the way the therapist thinks the client should see them?\n"
            "• Assess the cognitive shift following an intervention?"
        ),
        "anchors": {
            0: "If the therapist made no attempt to help the client achieve a cognitive shift",
            1: "If the therapist completed one or more but not all of the above items",
            2: "If the therapist completed all three of the above items",
            3: "If the therapist completed all three of the above items PLUS… Throughout the session, skillfully utilized the process of discovery to help the client arrive at their own conclusions; Assess the potential impact of the cognitive shift on the client's emotions and behaviors.",
        },
    },
    {
        "number": 8, "name": "Focus on Key Cognitions and Behaviors", "key": "item_8_focus_cognitions_behaviors", "short_label": "Focus on Key\nCog. & Beh.",
        "criteria": (
            "Did the therapist...\n"
            "• Focus on specific cognitions, images, sensations, emotions, behaviors, and or meanings about aspirations "
            "or challenges associated with the sessions Agenda item(s)?"
        ),
        "anchors": {
            0: "If the therapist did not focus on any particular item during the session",
            1: "If the therapist focused on an issue that was unrelated to Agenda items or was unable to elicit specific cognitions, images, sensations, emotions, behaviors, and/or meanings related to Agenda items",
            2: "If the therapist focused on specific cognitions, images, sensations, emotions, behaviors, and/or meanings about aspirations or challenges associated with the sessions Agenda items",
            3: "If the therapist completed the above item PLUS… The items(s) were the most relevant cognitions, images, sensations, emotions, and/or meanings that held greatest promise for a positive impact on the client's aspirations or challenges related to the sessions agenda item(s).",
        },
    },
    {
        "number": 9, "name": "Strategy for Change", "key": "item_9_strategy_for_change", "short_label": "Strategy\nfor Change",
        "criteria": (
            "Did the therapist...\n"
            "• Discuss evidence-based (CBT) techniques as part of an overall strategy for change with the client?\n"
            "• Select and use at least one identifiable evidence-based technique that was appropriate for the agenda item being addressed?"
        ),
        "anchors": {
            0: "If the therapist did not appear to have any strategy that incorporated use of evidence-based (CBT) techniques",
            1: "If the therapist appeared to have a strategy that did not include use of an appropriate evidence-based (CBT) technique",
            2: "If the therapist discussed an overall strategy for change with the client and used at least one appropriate evidence-based (CBT) technique",
            3: "If the therapist completed both of the items above PLUS… The therapist explained the rationale for use of the technique; Offered other options (if applicable); Obtained the client's agreement to participate in use of the techniques.",
        },
    },
    {
        "number": 10, "name": "Application of CBT Technique", "key": "item_10_application_cbt_technique", "short_label": "Application\nof CBT Tech.",
        "criteria": (
            "Did the therapist...\n"
            "• Apply a CBT technique with sufficient skill that the technique was recognizable?\n"
            "• Apply a CBT technique in such a way that it would likely facilitate change in a motivated client?"
        ),
        "anchors": {
            0: "If the therapist attempts to apply a CBT technique was not done with sufficient skill that it was recognizable",
            1: "If the therapist achieved one of the above items but not the other one",
            2: "If the therapist performed the technique with sufficient skill that it accomplished both of the above items",
            3: "If the therapist accomplished both of the above items PLUS The therapist demonstrated good familiarity with the technique; The therapist was comfortable applying the technique; The therapist applied the technique in a technically correct manner (i.e. as the technique is described in the literature).",
        },
    },
    {
        "number": 11, "name": "Action Plan", "key": "item_11_action_plan", "short_label": "Action\nPlan",
        "criteria": (
            "Did the therapist...\n"
            "• Review the Action Plan from the previous session?\n"
            "• Ask the client to provide input/agreement or incorporate spontaneously offered ideas into the development of a new Action Plan?\n"
            "• Develop an Action Plan based on work done in the current session [and/or continued from a previous session, if applicable] "
            "that, if completed, the Action Plan would answer a question, or help the client to better cope, develop a new skill, or improve their relationships?"
        ),
        "anchors": {
            0: "If the therapist did not complete any of the above items",
            1: "If the therapist completed one or more but not all of the items listed above",
            2: "If the therapist completed all three of the items listed above",
            3: "If the therapist completed all of the items listed above PLUS… The therapist ensured that the client knew what to do, was capable of doing it, and it was specified when, where, how often, and how long to do the Action Plan; and The therapist assessed the reasonable likelihood that the client would complete the Action Plan; and The therapist addressed any challenges or obstacles that would potentially reduce the likelihood of the client completing the Action Plan.",
        },
    },
]

ITEM_KEYS   = [it["key"]         for it in CTSR_ITEMS]
ITEM_LABELS = [it["short_label"] for it in CTSR_ITEMS]

print(f"CTRS-R items loaded: {len(CTSR_ITEMS)}")
for it in CTSR_ITEMS:
    print(f"  {it['number']:>2}. {it['name']}")

## Async DeepSeek Client

In [ ]:
ds_async = AsyncOpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

SEM = asyncio.Semaphore(API_CONCURRENCY)

print(f"Async DeepSeek client initialised (semaphore={API_CONCURRENCY}).")

## Conversation Engine

In [ ]:
def strip_thinking(text: str) -> str:
    """Strip Qwen3 <think>...</think> chain-of-thought blocks from model output."""
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return cleaned.strip()


def to_hf_messages(turns: list[dict], system_prompt: str) -> list[dict]:
    """Convert shared turn list to HF chat format (therapist POV)."""
    messages = [{"role": "system", "content": system_prompt}]
    for t in turns:
        role = "user" if t["role"] == "patient" else "assistant"
        messages.append({"role": role, "content": t["content"]})
    return messages


def to_patient_messages(turns: list[dict], system_prompt: str) -> list[dict]:
    """Convert shared turn list to chat format (patient POV)."""
    messages = [{"role": "system", "content": system_prompt}]
    for t in turns:
        role = "user" if t["role"] == "therapist" else "assistant"
        messages.append({"role": role, "content": t["content"]})
    return messages


def therapist_turn(model, tokenizer, turns: list[dict], system_prompt: str) -> str:
    """Generate therapist's next utterance on GPU (synchronous)."""
    messages = to_hf_messages(turns, system_prompt)
    if len(messages) == 1:
        messages.append({"role": "user", "content": "Please begin the session."})

    try:
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    stop_ids = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")})
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=THERAPIST_MAX_TOKENS,
            do_sample=True, temperature=0.7, top_p=0.9, top_k=50,
            eos_token_id=stop_ids,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return strip_thinking(raw)


async def patient_turn_async(client: AsyncOpenAI, vignette: dict, turns: list[dict]) -> str:
    """Generate patient's next utterance via DeepSeek (async)."""
    messages = to_patient_messages(turns, vignette["system_prompt"])
    async with SEM:
        resp = await client.chat.completions.create(
            model=PATIENT_MODEL,
            messages=messages,
            max_tokens=PATIENT_MAX_TOKENS,
            temperature=0.85,
            extra_body={"thinking": {"type": "disabled"}},
        )
    return resp.choices[0].message.content.strip()


async def should_end_session_async(client: AsyncOpenAI, turns: list[dict]) -> bool:
    """Moderator agent: checks if session should end (async)."""
    transcript_text = "\n".join(
        f"{'Therapist' if t['role'] == 'therapist' else 'Patient'}: {t['content']}"
        for t in turns
    )
    prompt = (
        "The following is a transcript of a CBT therapy session between a therapist and a patient:\n\n"
        f"{transcript_text}\n\n"
        "You must determine whether this session has FULLY reached a natural stopping point.\n"
        "Err on the side of letting the session continue — only end it when you are confident\n"
        "that ALL of the following have been CLEARLY and THOROUGHLY accomplished:\n\n"
        "1. The therapist has deeply explored the patient's core concerns (not just surface-level).\n"
        "2. The therapist has applied at least one specific, identifiable CBT technique or intervention\n"
        "   (e.g. thought record, behavioural experiment, Socratic questioning leading to cognitive shift).\n"
        "3. The therapist has collaboratively developed a concrete homework/action plan with the patient\n"
        "   (vague suggestions like 'think about it' do NOT count).\n"
        "4. The therapist has explicitly begun wrapping up (e.g. summarising the session, asking for\n"
        "   final feedback, or saying goodbye).\n"
        "5. The patient has no major unaddressed concerns remaining.\n\n"
        "The session should DEFINITELY continue if ANY of these are true:\n"
        "- The therapist is still in the middle of exploring a topic or concern\n"
        "- No specific CBT technique has been clearly applied yet\n"
        "- No concrete action plan or homework has been discussed\n"
        "- The therapist has not yet begun to wrap up\n"
        "- The patient has raised a concern that hasn't been adequately addressed\n"
        "- The conversation is still in the early/middle assessment phase\n\n"
        "When in doubt, answer 'No' (continue the session).\n\n"
        "Question: Has this therapy session FULLY reached a natural stopping point? "
        "Answer with ONLY 'Yes' or 'No'."
    )
    async with SEM:
        resp = await client.chat.completions.create(
            model=MODERATOR_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=3,
            temperature=0.0,
            extra_body={"thinking": {"type": "disabled"}},
        )
    answer = resp.choices[0].message.content.strip().lower()
    return answer.startswith("yes")


async def run_all_interviews(
    model, tokenizer, client: AsyncOpenAI, vignettes: list[dict],
    system_prompt: str,
) -> list[dict]:
    """
    Run all vignettes in lockstep.
    Each round: sequential GPU therapist turns, then concurrent API patient + moderator.
    """
    n = len(vignettes)
    all_turns: list[list[dict]] = [[] for _ in range(n)]
    active = list(range(n))

    # Round 0: therapist opening
    print(f"  Round 0: therapist opening for {len(active)} vignettes...", flush=True)
    for idx in active:
        t_msg = therapist_turn(model, tokenizer, all_turns[idx], system_prompt)
        all_turns[idx].append({"role": "therapist", "content": t_msg})
    print(f"    done ({len(active)} openings)")

    # Rounds 1..MAX_TURNS-1
    for round_num in range(1, MAX_TURNS):
        if not active:
            break

        # Patient turns — all concurrent
        print(f"  Round {round_num}: patient×{len(active)}...", end=" ", flush=True)
        patient_tasks = [
            patient_turn_async(client, vignettes[idx], all_turns[idx])
            for idx in active
        ]
        patient_responses = await asyncio.gather(*patient_tasks)
        for i, idx in enumerate(active):
            all_turns[idx].append({"role": "patient", "content": patient_responses[i]})

        # Therapist turns — sequential on GPU
        print(f"therapist×{len(active)}...", end=" ", flush=True)
        for idx in active:
            t_msg = therapist_turn(model, tokenizer, all_turns[idx], system_prompt)
            all_turns[idx].append({"role": "therapist", "content": t_msg})

        # Moderator checks — concurrent for eligible conversations
        eligible = [
            idx for idx in active
            if (len(all_turns[idx]) + 1) // 2 >= MIN_TURNS
        ]
        ended = set()
        if eligible:
            print(f"moderator×{len(eligible)}...", end=" ", flush=True)
            mod_tasks = [
                should_end_session_async(client, all_turns[idx])
                for idx in eligible
            ]
            mod_results = await asyncio.gather(*mod_tasks)
            for i, idx in enumerate(eligible):
                if mod_results[i]:
                    ended.add(idx)

        active = [idx for idx in active if idx not in ended]
        print(f"ended={len(ended)}, active={len(active)}")

    results = []
    for idx in range(n):
        results.append({
            "vignette": vignettes[idx]["name"],
            "turns": all_turns[idx],
        })
    return results


print("Conversation engine defined.")

## Critic Agent

Evaluates the therapist's performance and gives detailed, actionable feedback using the CTRS-R rubric.

In [ ]:
def build_critic_prompt(vignette: dict, turns: list[dict]) -> str:
    """Build a comprehensive critic prompt that scores all 11 CTRS-R items and gives feedback."""
    transcript_text = "\n".join(
        f"{'Therapist' if t['role'] == 'therapist' else 'Patient'}: {t['content']}"
        for t in turns
    )

    rubric_text = ""
    for item in CTSR_ITEMS:
        anchors_text = "\n".join(f"      {k} = {v}" for k, v in item["anchors"].items())
        rubric_text += (
            f"  Item {item['number']}: {item['name']} (key: {item['key']})\n"
            f"    Criteria:\n{item['criteria']}\n"
            f"    Score anchors:\n{anchors_text}\n\n"
        )

    return (
        "You are a CBT clinical supervisor providing detailed feedback to a therapist-in-training.\n"
        "Evaluate the therapist's performance using the Cognitive Therapy Rating Scale – Revised (CTRS-R).\n\n"
        f"Patient background:\n{vignette['background']}\n\n"
        f"Session transcript:\n{transcript_text}\n\n"
        f"CTRS-R Rubric:\n{rubric_text}\n"
        "Score the therapist on ALL 11 items. For each item, provide reasoning "
        "with specific transcript references, a score, and actionable improvement suggestions.\n\n"
        "Output ONLY a JSON object with this exact structure:\n"
        "{\n"
        '  "item_1_agenda": {"reasoning": "...", "score": N, "improvement": "..."},\n'
        '  "item_2_feedback": {"reasoning": "...", "score": N, "improvement": "..."},\n'
        '  "item_3_understanding": {"reasoning": "...", "score": N, "improvement": "..."},\n'
        '  "item_4_interpersonal_effectiveness": {"reasoning": "...", "score": N, "improvement": "..."},\n'
        '  "item_5_collaboration": {"reasoning": "...", "score": N, "improvement": "..."},\n'
        '  "item_6_pacing": {"reasoning": "...", "score": N, "improvement": "..."},\n'
        '  "item_7_guided_discovery": {"reasoning": "...", "score": N, "improvement": "..."},\n'
        '  "item_8_focus_cognitions_behaviors": {"reasoning": "...", "score": N, "improvement": "..."},\n'
        '  "item_9_strategy_for_change": {"reasoning": "...", "score": N, "improvement": "..."},\n'
        '  "item_10_application_cbt_technique": {"reasoning": "...", "score": N, "improvement": "..."},\n'
        '  "item_11_action_plan": {"reasoning": "...", "score": N, "improvement": "..."}\n'
        "}\n\n"
        "Rules:\n"
        "- Output ONLY the JSON object, no other text.\n"
        "- Each score must be an integer 0–3.\n"
        "- Each reasoning must be concise (2–4 sentences) with specific transcript references.\n"
        "- Each improvement must be a specific, actionable suggestion for what to do differently."
    )


def format_critic_feedback(raw_json: str) -> str:
    """Format structured critic JSON into readable feedback for the system prompt."""
    try:
        data = json.loads(raw_json)
    except (json.JSONDecodeError, ValueError):
        return raw_json  # fallback to raw text

    parts = []
    for item in CTSR_ITEMS:
        key = item["key"]
        if key in data and isinstance(data[key], dict):
            entry = data[key]
            parts.append(
                f"Item {item['number']} ({item['name']}): {entry.get('score', '?')}/3\n"
                f"  Reasoning: {entry.get('reasoning', 'N/A')}\n"
                f"  Improvement: {entry.get('improvement', 'N/A')}"
            )

    return "\n".join(parts)


async def get_critic_feedback(
    client: AsyncOpenAI, vignette: dict, turns: list[dict],
) -> str:
    """Get critic feedback for a single dialogue (async). Returns formatted feedback."""
    prompt = build_critic_prompt(vignette, turns)
    async with SEM:
        resp = await client.chat.completions.create(
            model=CRITIC_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=CRITIC_MAX_TOKENS,
            temperature=0.3,
            extra_body={"thinking": {"type": "disabled"}},
        )
    raw = resp.choices[0].message.content.strip()

    # Strip code fences if present
    cleaned = raw
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)

    return format_critic_feedback(cleaned)


async def get_all_critic_feedback(
    client: AsyncOpenAI, transcripts: list[dict], vignettes: list[dict],
) -> list[str]:
    """Get critic feedback for all transcripts concurrently."""
    vignette_map = {v["name"]: v for v in vignettes}
    tasks = [
        get_critic_feedback(client, vignette_map[t["vignette"]], t["turns"])
        for t in transcripts
    ]
    print(f"  Firing {len(tasks)} critic calls (semaphore={API_CONCURRENCY})...", flush=True)
    results = await asyncio.gather(*tasks)
    print(f"  All {len(tasks)} critic calls complete.")
    return results


print("Critic agent defined.")

## Build Refined System Prompt

For each refinement iteration, the therapist gets the base system prompt plus
all previous attempt transcripts and critic feedback.

In [ ]:
def build_refined_system_prompt(
    base_system: str,
    previous_attempts: list[dict],  # [{"turns": [...], "feedback": "..."}]
) -> str:
    """
    Build system prompt that includes previous attempts and feedback.

    Format:
        [base system prompt]

        PREVIOUS ATTEMPT 1:
        [dialogue transcript]

        CRITIC FEEDBACK 1:
        [feedback]

        ...

        Use the lessons from the previous attempts and feedback.
        Start a new conversation with the patient from the beginning.
    """
    parts = [base_system]

    for i, attempt in enumerate(previous_attempts, 1):
        transcript_text = "\n".join(
            f"{'Therapist' if t['role'] == 'therapist' else 'Patient'}: {t['content']}"
            for t in attempt["turns"]
        )
        parts.append(f"\nPREVIOUS ATTEMPT {i}:\n{transcript_text}")
        parts.append(f"\nCRITIC FEEDBACK {i}:\n{attempt['feedback']}")

    parts.append(
        "\nUse the lessons from the previous attempts and feedback."
        "\nStart a new conversation with the patient from the beginning."
    )

    return "\n".join(parts)


print("Refined system prompt builder defined.")

## Load SFT Model

In [ ]:
def load_therapist_model(model_path: str, base_model_name: str | None = None):
    """Load a therapist model for inference."""
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    path = Path(model_path)

    is_peft = (path / "adapter_config.json").exists()
    if is_peft:
        from peft import PeftModel
        assert base_model_name, "base_model_name required when loading a PEFT adapter"
        print(f"  Detected PEFT adapter — loading base + adapter and merging…")
        tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        base = AutoModelForCausalLM.from_pretrained(
            base_model_name, torch_dtype=dtype, device_map="auto", trust_remote_code=True
        )
        model = PeftModel.from_pretrained(base, model_path)
        model = model.merge_and_unload()
    else:
        print(f"  Loading full model from {model_path}…")
        tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=dtype, device_map="auto", trust_remote_code=True
        )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    total_params = sum(p.numel() for p in model.parameters()) / 1e9
    print(f"  Loaded {total_params:.1f}B parameters.")
    return model, tokenizer


print("Loading SFT model...")
sft_model, sft_tok = load_therapist_model(
    str(SFT_MODEL_PATH), base_model_name=BASE_MODEL_NAME
)
print("SFT model loaded.")

## Self-Play Loop

For each outer loop iteration:
1. Generate 112 dialogues
2. Get critic feedback (parallelized)
3. Regenerate dialogues with feedback in system prompt
4. Repeat critic→regenerate `N_REFINEMENT_CYCLES` times
5. Keep only the final refined dialogues

In [ ]:
async def self_play_iteration(
    model, tokenizer, client: AsyncOpenAI, vignettes: list[dict],
    n_refinement_cycles: int,
) -> list[dict]:
    """
    Run one full self-play iteration:
    - Generate initial dialogues
    - For each refinement cycle: critic feedback → regenerate
    - Return the final refined dialogues
    """
    n = len(vignettes)

    # Track all previous attempts per vignette: [{"turns": [...], "feedback": "..."}]
    history: list[list[dict]] = [[] for _ in range(n)]

    # ── Initial generation (no feedback yet) ──────────────────────────────────
    print(f"\n{'='*60}")
    print(f"INITIAL GENERATION ({n} vignettes)")
    print(f"{'='*60}")
    t0 = time.perf_counter()
    transcripts = await run_all_interviews(
        model, tokenizer, client, vignettes, THERAPIST_SYSTEM
    )
    elapsed = time.perf_counter() - t0
    print(f"  Initial generation done in {elapsed:.1f}s")

    # ── Refinement cycles ─────────────────────────────────────────────────────
    for cycle in range(n_refinement_cycles):
        print(f"\n{'='*60}")
        print(f"REFINEMENT CYCLE {cycle + 1}/{n_refinement_cycles}")
        print(f"{'='*60}")

        # Get critic feedback for current transcripts (parallelized)
        print(f"\n  Getting critic feedback...")
        t0 = time.perf_counter()
        feedbacks = await get_all_critic_feedback(client, transcripts, vignettes)
        elapsed = time.perf_counter() - t0
        print(f"  Critic feedback done in {elapsed:.1f}s")

        # Store this attempt in history
        for idx in range(n):
            history[idx].append({
                "turns": transcripts[idx]["turns"],
                "feedback": feedbacks[idx],
            })

        # Build refined system prompts for each vignette
        refined_prompts = [
            build_refined_system_prompt(THERAPIST_SYSTEM, history[idx])
            for idx in range(n)
        ]

        # Regenerate all dialogues with feedback
        print(f"\n  Regenerating dialogues with feedback...")
        t0 = time.perf_counter()

        # Run interviews with per-vignette system prompts
        # We need to run them individually since each has a different system prompt
        new_transcripts = []
        all_turns_new: list[list[dict]] = [[] for _ in range(n)]
        active = list(range(n))

        # Round 0: therapist opening
        print(f"    Round 0: therapist opening for {len(active)} vignettes...", flush=True)
        for idx in active:
            t_msg = therapist_turn(model, tokenizer, all_turns_new[idx], refined_prompts[idx])
            all_turns_new[idx].append({"role": "therapist", "content": t_msg})
        print(f"      done ({len(active)} openings)")

        # Rounds 1..MAX_TURNS-1
        for round_num in range(1, MAX_TURNS):
            if not active:
                break

            # Patient turns — concurrent
            print(f"    Round {round_num}: patient×{len(active)}...", end=" ", flush=True)
            patient_tasks = [
                patient_turn_async(client, vignettes[idx], all_turns_new[idx])
                for idx in active
            ]
            patient_responses = await asyncio.gather(*patient_tasks)
            for i, idx in enumerate(active):
                all_turns_new[idx].append({"role": "patient", "content": patient_responses[i]})

            # Therapist turns — sequential on GPU
            print(f"therapist×{len(active)}...", end=" ", flush=True)
            for idx in active:
                t_msg = therapist_turn(model, tokenizer, all_turns_new[idx], refined_prompts[idx])
                all_turns_new[idx].append({"role": "therapist", "content": t_msg})

            # Moderator checks — concurrent
            eligible = [
                idx for idx in active
                if (len(all_turns_new[idx]) + 1) // 2 >= MIN_TURNS
            ]
            ended = set()
            if eligible:
                print(f"moderator×{len(eligible)}...", end=" ", flush=True)
                mod_tasks = [
                    should_end_session_async(client, all_turns_new[idx])
                    for idx in eligible
                ]
                mod_results = await asyncio.gather(*mod_tasks)
                for i, idx in enumerate(eligible):
                    if mod_results[i]:
                        ended.add(idx)

            active = [idx for idx in active if idx not in ended]
            print(f"ended={len(ended)}, active={len(active)}")

        # Build transcripts from this cycle
        transcripts = []
        for idx in range(n):
            transcripts.append({
                "vignette": vignettes[idx]["name"],
                "turns": all_turns_new[idx],
            })

        elapsed = time.perf_counter() - t0
        print(f"  Regeneration done in {elapsed:.1f}s")

    # Return the final refined transcripts
    return transcripts


print("Self-play iteration defined.")

## Run Self-Play

In [ ]:
all_refined_dialogues: list[dict] = []

for outer_loop in range(N_OUTER_LOOPS):
    print(f"\n{'#'*60}")
    print(f"OUTER LOOP {outer_loop + 1}/{N_OUTER_LOOPS}")
    print(f"{'#'*60}")

    t0 = time.perf_counter()
    refined = await self_play_iteration(
        sft_model, sft_tok, ds_async, VIGNETTES,
        n_refinement_cycles=N_REFINEMENT_CYCLES,
    )
    elapsed = time.perf_counter() - t0

    all_refined_dialogues.extend(refined)
    print(f"\n  Outer loop {outer_loop + 1} complete in {elapsed:.1f}s")
    print(f"  Collected {len(refined)} refined dialogues (total: {len(all_refined_dialogues)})")

print(f"\n{'='*60}")
print(f"SELF-PLAY COMPLETE")
print(f"Total refined dialogues: {len(all_refined_dialogues)}")
print(f"{'='*60}")

## Convert to SFT Format & Save

In [ ]:
def dialogue_to_sft_format(dialogue: dict) -> dict:
    """
    Convert a refined dialogue into SFT training format.
    Format: {"messages": [{"role": "system", ...}, {"role": "assistant", ...}, {"role": "user", ...}, ...]}
    Therapist = assistant, Patient = user.
    """
    messages = [{"role": "system", "content": THERAPIST_SYSTEM}]
    for turn in dialogue["turns"]:
        if turn["role"] == "therapist":
            messages.append({"role": "assistant", "content": turn["content"]})
        else:
            messages.append({"role": "user", "content": turn["content"]})
    return {"messages": messages}


# Convert all refined dialogues
sft_records = [dialogue_to_sft_format(d) for d in all_refined_dialogues]

# Save to JSONL
with open(SELF_PLAY_JSONL, "w") as f:
    for record in sft_records:
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(sft_records)} SFT records → {SELF_PLAY_JSONL}")
print(f"File size: {SELF_PLAY_JSONL.stat().st_size / 1024:.1f} KB")

# Show stats
turn_counts = [len(d["turns"]) for d in all_refined_dialogues]
print(f"\nDialogue stats:")
print(f"  Total dialogues: {len(all_refined_dialogues)}")
print(f"  Avg turns: {sum(turn_counts) / len(turn_counts):.1f}")
print(f"  Min turns: {min(turn_counts)}")
print(f"  Max turns: {max(turn_counts)}")

In [ ]:
# Preview a sample
print("Sample refined dialogue (first 4 turns):")
print(f"  Vignette: {all_refined_dialogues[0]['vignette']}")
for turn in all_refined_dialogues[0]["turns"][:4]:
    role = "Therapist" if turn["role"] == "therapist" else "Patient"
    print(f"  [{role}]: {turn['content'][:150]}...")

# Cleanup
del sft_model, sft_tok
torch.cuda.empty_cache()
print("\nModel unloaded. Done.")